# TTS model benchmark on OCI Generative AI

Measures speed, latency, concurrency behaviour and long-input quality of a text-to-speech
endpoint (for example OmniVoice imported on a dedicated AI cluster) in English and Arabic.

Runs as-is in an **OCI Data Science notebook session** (resource principal auth) or on a laptop
with an `~/.oci/config` profile. Each step shells out to the scripts in the parent folder, so
everything here can also be run from a terminal.

In [ ]:
%cd ..
!pip install -q -r requirements.txt

## 1. Configure

Set the endpoint OCID and compartment. In a Data Science notebook session the resource principal is
detected automatically; the session's dynamic group needs `use generative-ai-inference` in that compartment.
On a laptop set `OCI_AUTH=api_key` and `OCI_CONFIG_PROFILE`.

In [ ]:
import os
os.environ["TTS_MODEL"] = "ocid1.generativeaiendpoint.oc1.<region>.<endpoint-id>"   # the endpoint OCID
os.environ["GENAI_COMPARTMENT_ID"] = "ocid1.compartment.oc1..<compartment-id>"
# os.environ["OCI_AUTH"] = "api_key"; os.environ["OCI_CONFIG_PROFILE"] = "DEFAULT"   # laptop
LABEL = "H100_X1"   # name the run after the hosting shape so runs can be compared

## 2. Smoke test (about two minutes)

Two lengths, streaming, three requests each, English and Arabic. Confirms auth and the route.

In [ ]:
!python tts_bench.py --suite quick --label {LABEL}

## 3. The real runs

Pick what you need. Each writes its own folder under `results/`.

| suite | question it answers | time on one GPU |
|---|---|---|
| `latency` | how do TTFA and E2E grow with input length, with and without streaming, and with client-side chunking? | 20-40 min |
| `long-text` | at what length does single-pass output degrade? saves one clip per length for Whisper scoring | 10-15 min |
| `concurrency` | how many parallel requests before latency climbs and throughput flattens? | 10-15 min |
| `full` | all three | about an hour |

In [ ]:
!python tts_bench.py --suite latency --label {LABEL}

In [ ]:
!python tts_bench.py --suite long-text --label {LABEL}

In [ ]:
!python tts_bench.py --suite concurrency --label {LABEL}

## 4. Look at the results

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, Markdown, display

runs = sorted(Path("results").glob(f"{LABEL}_*"))
run = runs[-1]
print("latest run:", run)
summary = pd.read_csv(run / "summary.csv")
cols = ["suite", "lang", "text", "mode", "target_s", "chars", "concurrency", "n", "audio_s_mean",
        "ttfa_p50", "ttfa_p90", "e2e_p50", "e2e_p90", "e2e_p99", "rtf_p50", "error_rate", "req_per_min", "audio_s_per_s"]
summary[cols].round(2)

In [ ]:
!python plots.py {run}
for png in sorted((run / "charts").glob("*.png")):
    display(Image(filename=str(png)))

## 5. Intelligibility: where does the audio break?

Latency numbers do not show the babble that single-pass synthesis produces past a certain length.
Transcribe the saved samples with Whisper and compare them to the text that was sent. WER under 5 %
is good audio; a jump to tens of percent marks the breaking point. `faster-whisper` runs on CPU;
on a GPU session it uses CUDA automatically. The first run downloads the model (about 1.6 GB for
`large-v3-turbo`; use `--model small` for a quick look).

In [ ]:
!pip install -q faster-whisper
!python wer.py {run}
pd.read_csv(run / "wer.csv").pivot_table(index=["lang", "text", "target_s"], columns="mode", values="wer").round(3)

In [ ]:
# redraw the charts so the WER panel is included, and show the written report
!python plots.py {run}
display(Image(filename=str(run / "charts" / "wer_vs_length.png")))
display(Markdown((run / "summary.md").read_text()))

## 6. Compare hosting shapes

Run the same suite with a different `LABEL` against an endpoint on another shape, then put the
summaries side by side:

```python
a = pd.read_csv("results/A10_X1_<time>/summary.csv")
b = pd.read_csv("results/H100_X1_<time>/summary.csv")
key = ["suite", "lang", "text", "mode", "target_s", "concurrency"]
a.merge(b, on=key, suffixes=("_a10", "_h100"))[key + ["e2e_p50_a10", "e2e_p50_h100", "req_per_min_a10", "req_per_min_h100"]]
```